In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from fastai.vision import *

import IPython.display as ipd
from tqdm import tqdm_notebook

In [ ]:
import re
pat = re.compile(r'.*/(Female|Male)_[\w\d]+\.jpg')



In [ ]:
from pathlib import Path
import numpy as np

path_img = "../../../Data_mfc/CTU/3_mfcc_3d"
fnames = list(Path(path_img).rglob('*.jpg')) + list(Path(path_img).rglob('*.jpeg')) + list(Path(path_img).rglob('*.png'))
np.random.seed(42)
np.random.shuffle(fnames)
fnames[:5]


In [ ]:
from fastai.vision.all import *
# Tạo DataBlock
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=RegexLabeller(pat)
)

# Tạo DataLoaders
dls = dblock.dataloaders(path_img, bs=64)


# Hiển thị batch và classes
dls.show_batch()
print(dls.vocab)



In [ ]:
# Replace the old show_batch code with the following
dls.show_batch(nrows=3, ncols=3, figsize=(7,6))


In [ ]:

learn = cnn_learner(dls, models.resnet34, metrics=error_rate)

# Provide the correct file path to the exported learner file
learner_path = "mfcc_3d.pkl"

# Load the learner model from the exported file
learn.model = load_learner(learner_path).model


In [ ]:
interp = ClassificationInterpretation.from_learner(learn, ds_idx=1)  # Assuming validation data is at index 1



In [ ]:
losses, idxs = interp.top_losses()  # only uses data from the validation set

In [ ]:
interp.plot_top_losses(9, figsize=(10, 6))


In [ ]:
interp.plot_confusion_matrix(figsize=(5,6), dpi=60)

In [ ]:
cmat = interp.confusion_matrix()
precision = cmat[0][0] / (cmat[0][0] + cmat[1][0])  # true positives / (true positives + false positives = positive predictions) 
recall = cmat[0][0] / (cmat[0][0] + cmat[0][1])  # true positives / (true positives + false negatives = positive ground truths)
f1_score = 2 / (1 / precision + 1 / recall)
accuracy = (cmat[0][0] + cmat[1][1]) / (cmat[0][0] + cmat[0][1] + cmat[1][0] + cmat[1][1])


In [ ]:
precision

In [ ]:
recall

In [ ]:
f1_score

In [ ]:
accuracy

In [ ]:
from sklearn.metrics import classification_report

# Get predictions and true labels
preds, targs = learn.get_preds(ds_idx=1)

# Convert predictions and true labels to class indices
preds_classes = preds.argmax(dim=1)
targs_classes = targs.int()

# Convert class indices to class labels using the DataLoader's vocab
preds_labels = [dls.vocab[i] for i in preds_classes]
targs_labels = [dls.vocab[i] for i in targs_classes]

# Generate classification report
class_report = classification_report(targs_labels, preds_labels, digits=4)  # Specify digits parameter
error_rate = 1.0 - (cmat[0][0] + cmat[1][1]) / len(targs_labels)
print(error_rate)
print(class_report)
